In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd

from bs4 import BeautifulSoup
import requests
import random
import string

import datetime
from selenium import webdriver
from time import sleep
import os
from collections import defaultdict
from DrissionPage import ChromiumPage, ChromiumOptions, SessionPage
from DrissionPage.errors import ElementNotFoundError


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'IT CONSOB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
#writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running IT CONSOB Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder



chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

chromeOptions.add_experimental_option("excludeSwitches", ["enable-automation"])

user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"

chromeOptions.add_argument(f"user-agent={user_agent}")



driver = webdriver.Chrome(options=chromeOptions)

# driver.maximize_window()
# options = ChromiumOptions()
# options.set_download_path(tempfolder)
# options.incognito(True)

# driver = ChromiumPage(options)


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict={

        regulatorName+' 1': 'https://www.consob.it/web/consob-and-its-activities/class-1-investment-firms-authorised-in-other-eu-countries-with-branches-in-italy',    

        regulatorName+' 2': 'https://www.consob.it/web/consob-and-its-activities/class-1-investment-firms-authorised-in-other-eu-countries-without-branches-in-italy', 

        regulatorName+' 3': 'https://www.consob.it/web/consob-and-its-activities/companies-of-non-eu-authorized-to-operate-in-italy-with-branches', 

        regulatorName+' 4': 'https://www.consob.it/web/consob-and-its-activities/companies-non-eu-authorized-in-italy-without-branches', 

        regulatorName+' 5': 'https://www.consob.it/web/consob-and-its-activities/register-of-italian-investment-firms-sims-', 

        regulatorName+' 6': 'https://www.consob.it/web/consob-and-its-activities/investment-firms-with-branches', 

        regulatorName+' 7': 'https://www.consob.it/web/consob-and-its-activities/listed-companies', 

        regulatorName+' 8': 'https://www.consob.it/web/consob-and-its-activities/mtf-authorised-consob',

        }

Typology ={

        regulatorName+' 1': 'Class 1 Investment firms authorised in other EU countries with branches in Italy',    

        regulatorName+' 2': 'Class 1 investment firms authorised in other EU countries without branches in Italy', 

        regulatorName+' 3': 'Companies of non-EU countries other than banks authorized by Consob to operate in Italy with branches', 

        regulatorName+' 4': 'Companies of non-EU countries other than banks authorized by Consob to operate in Italy without branches', 

        regulatorName+' 5': 'Italian investment firms (SIMs)', 

        regulatorName+' 6': 'List of Investment Firms authorised in other EU states with branches in Italy', 

        regulatorName+' 7': 'Listed Companies', 

        regulatorName+' 8': 'Markets', 
        
        }

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

def clean_text(value):
    return value.replace('\n', '').replace('\t', '').strip()

def safe_get(record, key, idx=0, default=''):
    key = key.lower().strip()
    values = record.get(key)

    if values is None:
        for actual_key, actual_values in record.items():
            if key in actual_key:
                values = actual_values
                break

    if not values or len(values) <= idx:
        return default
    return values[idx]



In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_{Typology[reg]}")
    #response = session.get(regdict[reg], headers=headers,timeout=30,verify=False)

    if reg != 'IT CONSOB 7':

        # driver = webdriver.Chrome(options=chromeOptions)
        # driver.maximize_window()
        driver.get(regdict[reg])
        raw = {
            "ssresp": "0",
            "jsrecvd": "true",
            "__uzmaj": "0192c3fe-5e00-4c10-9bc5-0aafba391ce8",
            "__uzmbj": "1762341890",
            "__uzmcj": "7337110995576",
            "__uzmdj": "1773235561",
            "jsbd2": "4ce14fe9-ch6x-88d6-11cd-4d9b28897a00"
        }

        cookies = [{"name": k, "value": v, "domain": ".consob.it", "path": "/"} for k, v in raw.items()]
        driver.set.cookies(cookies)
        driver.refresh()
        sleep(10)
        soup = BeautifulSoup(driver.html, "html.parser")
        tables = soup.find_all('table')
        print('Table found using selenium {}'.format(len(tables)))

        if reg != 'IT CONSOB 8' :
            records = []
            for table in soup.select("table"):
                data = defaultdict(list)
                for row in table.select("tr"):
                    cells = row.find_all(["th", "td"])
                    if len(cells) < 2:
                        continue
                    label = cells[0].get_text(strip=True).rstrip(":").lower()
                    value = cells[1].get_text(" ", strip=True).replace('\xa0', ' ')
                    data[label].append(value)
                records.append(data)

            for rec in records:
                name = safe_get(rec, 'investment firm')
                lei_code = safe_get(rec, 'lei')
                registered_office = safe_get(rec, 'office')

                head_city = clean_text(safe_get(rec, 'city'))
                branch_city = clean_text(safe_get(rec, 'city', idx=1))
                head_country = clean_text(safe_get(rec, 'country'))
                branch_country = clean_text(safe_get(rec, 'country', idx=1))
                branch_addr = clean_text(safe_get(rec, 'branch'))

                zip_mother_company = head_city.split(' ')[0] if head_city else ''
                zip_branch_company = branch_city.split(' ')[0] if branch_country else ''
                parts = head_city.split(' ') if head_city else []
                
                sqldict['Name'].append(name)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['Address_1 - Mother company'].append(registered_office)
                sqldict['Address_1'].append(branch_addr)
                sqldict['Zip'].append(zip_branch_company)
                try:
                    sqldict['City'].append(branch_city.split(' ')[1])
                    sqldict['Cntry'].append(branch_country)
                except:
                    sqldict['City'].append('')
                    sqldict['Cntry'].append('')
                if len(parts) == 3 and not parts[-1].startswith('('):
                    head_city_ = head_city.split(' ')[-1]
                    zip_ = ''.join(head_city.split(' ')[0:-1])
                    #print(zip_)
                elif len(parts) == 3 and parts[-1].startswith('('):
                    head_city_ = head_city.split(' ')[1]
                    zip_ = head_city.split(' ')[0]
                elif len(parts) == 2:
                    head_city_ = head_city.split(' ')[-1]
                    zip_mother_company = head_city.split(' ')[0]
                sqldict['City - Mother company'].append(head_city_)
                sqldict['Zip - Mother company'].append(zip_mother_company)
                sqldict['Cntry - Mother company'].append(head_country)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['ListName'].append(Typology[reg])

                sqldict = bourange_same_length_array(sqldict)

            # driver.quit()
    

        
        elif reg == 'IT CONSOB 8' :
            driver.get(regdict[reg])
            sleep(10)
            tds = tables[0].find('tbody').find_all('td')
            company_name = ''
            for index,td in enumerate(tds):
                if  td.find('strong'):
                    company_name = td.find('strong').text
                    #print(company_name)
                else:
                    
                    if 'text-align: center' in td.get('style'):
                        mic_code = td.text
                        #print(mic_code)
                        if mic_code.strip():
                            sqldict['InternalID_1'].append(mic_code)
                            sqldict['InternalID_1_type'].append('MIC Code')
                            sqldict['ListProcessDate'].append(processdate)
                    else:
                        mtf_name = td.text
                        #print(mtf_name)

                        sqldict['Name'].append(mtf_name)
                        sqldict['Name - Mother Company'].append(company_name)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict["ListName"].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)
            driver.quit()
    
    else:

        base_letter_url = "https://www.consob.it/web/consob-and-its-activities/listed-companies/list?startsWith="
        for letter in string.ascii_uppercase:
            letter_url = base_letter_url + letter
            print(f"Processing letter {letter} at {letter_url}")
            driver.get(letter_url)
            sleep(random.uniform(3, 6))
            if "we apologize for the inconvenience" in driver.html.lower():
                
                sleep(10)
                driver.get(letter_url)
                
            soup = BeautifulSoup(driver.html, 'html.parser')
            company_spans = soup.select("span.boxQuotataTitle")
            #print(f"Found {len(company_spans)} company names for letter {letter}")
            for span in company_spans:
                name_val = span.get_text(strip=True)
                # Append company info for CONSOB 7
                sqldict['Name'].append(name_val)
                
                sqldict['Address_1'].append("")
                sqldict['City'].append("")
                sqldict['Cntry'].append("")
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
# driver.quit()

[INFO] : Working 1/8 _(IT CONSOB 1)_Class 1 Investment firms authorised in other EU countries with branches in Italy


AttributeError: 'WebDriver' object has no attribute 'set'

In [ ]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
sqldict = bourange_same_length_array(sqldict)
df=pd.DataFrame(sqldict)
df.to_excel(filename,index=False)


sleep(3)
driver.quit()

In [ ]:
import os
import re
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

df = pd.read_excel('IT CONSOB SQL Ready 2026-03-11 14.40.43.xlsx')

def safe_filename(name):
    name = str(name).strip()
    return re.sub(r'[\\/:*?"<>|]+', '_', name) or "UNKNOWN"

def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont("Helvetica", 12)

    x, y = 72, 720
    max_lines_per_page = 33
    line_count = 0

    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20
        line_count += 1

        if line_count >= max_lines_per_page:
            c.showPage()
            c.setFont("Helvetica", 12)
            x, y = 72, 720
            line_count = 0

    c.save()

os.makedirs(tempfolder, exist_ok=True)

for list_code, group in df.groupby('ListCode', dropna=False):
    code = safe_filename(list_code)
    items = group['Name'].dropna().astype(str).tolist()
    if not items:
        continue
    pdf_path = os.path.join(tempfolder, f"IT_CONSOB SQL Ready 2026-03-11 14.40.43- {code}.pdf")
    export_list_to_pdf(items, pdf_path)
